# Module 03 — Notebook 03: Neuron Conductance

## Learning Objectives
By completing this notebook, you will:
1. Use `NeuronConductance` to find which input features activate specific neurons in ResNet-50
2. Identify the most and least important neurons for a prediction using Layer Conductance as a filter
3. Visualize what individual neurons "see" in the input image
4. Compare neuron selectivity across classes (class-specific vs. shared neurons)

## Prerequisites
- Module 03 Notebook 02 (Layer Conductance)
- Guide 02 (Conductance theory — Neuron Conductance section)

## Estimated Time: 15 minutes

---

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import io
import json
from PIL import Image

from captum.attr import LayerConductance, NeuronConductance

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(t.cpu() * std + mean, 0, 1)

model = models.resnet50(weights='IMAGENET1K_V1').to(DEVICE)
model.eval()
print("ResNet-50 loaded.")

In [ ]:
# Load ImageNet labels and dog image
LABELS_URL = (
    "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels"
    "/master/imagenet-simple-labels.json"
)
try:
    with urllib.request.urlopen(LABELS_URL) as resp:
        labels = json.loads(resp.read().decode())
except Exception:
    labels = [f"class_{i}" for i in range(1000)]
    labels[208] = "Labrador retriever"

DOG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg"
try:
    with urllib.request.urlopen(DOG_URL, timeout=10) as resp:
        raw = Image.open(io.BytesIO(resp.read())).convert('RGB')
except Exception:
    raw = Image.fromarray(np.random.randint(80, 200, (320, 320, 3), dtype=np.uint8))

input_tensor = preprocess(raw).unsqueeze(0).to(DEVICE)
img_np = denormalize(input_tensor.squeeze(0)).permute(1, 2, 0).numpy()
baseline = torch.zeros_like(input_tensor)

with torch.no_grad():
    logits = model(input_tensor)
    top_class = logits.argmax().item()

print(f"Prediction: {labels[top_class]} (class {top_class})")

## 1. Finding the Most Important Neurons

We use a two-step approach:
1. **Layer Conductance** → identify which neurons in `layer4` have the highest absolute conductance
2. **Neuron Conductance** → for each top neuron, compute which input features caused it to activate

This is efficient: we don't need Neuron Conductance for all 2048×7×7 = 100,352 neurons — just the top few.

In [ ]:
# Step 1: Layer Conductance to rank neurons
TARGET_LAYER = model.layer4[-1]
N_STEPS = 50
TOP_K = 6  # Number of top neurons to inspect

lc = LayerConductance(model, TARGET_LAYER)
layer_attr = lc.attribute(
    input_tensor,
    baselines=baseline,
    target=top_class,
    n_steps=N_STEPS
)
# layer_attr: (1, 2048, 7, 7)

print(f"Layer4 attribution shape: {layer_attr.shape}")
print(f"  Channels: {layer_attr.shape[1]}")
print(f"  Spatial:  {layer_attr.shape[2]}×{layer_attr.shape[3]}")
print(f"  Total neurons: {layer_attr.shape[1] * layer_attr.shape[2] * layer_attr.shape[3]:,}")

# Flatten and find top-K neurons by absolute conductance
C, H, W = layer_attr.shape[1], layer_attr.shape[2], layer_attr.shape[3]
attr_flat = layer_attr.abs().squeeze(0).view(-1)  # (C*H*W,)
top_flat_idxs = attr_flat.topk(TOP_K).indices

top_neurons = []
for flat_idx in top_flat_idxs:
    flat_idx = flat_idx.item()
    c = flat_idx // (H * W)
    hw = flat_idx % (H * W)
    h = hw // W
    w = hw % W
    cond = layer_attr[0, c, h, w].item()
    top_neurons.append((c, h, w, cond))

print(f"\nTop-{TOP_K} neurons by |conductance| in layer4:")
for i, (c, h, w, cond) in enumerate(top_neurons):
    print(f"  #{i+1}: channel={c:4d}, pos=({h},{w}), conductance={cond:+.5f}")

In [ ]:
# Step 2: Neuron Conductance for each top neuron
# Neuron Conductance returns input attribution: which input pixels caused
# this specific neuron to activate as it did?

nc = NeuronConductance(model, TARGET_LAYER)

neuron_maps = []
for c, h, w, cond in top_neurons:
    neuron_attr = nc.attribute(
        input_tensor,
        neuron_selector=(c, h, w),  # (channel, spatial_h, spatial_w)
        target=top_class,
        baselines=baseline,
        n_steps=N_STEPS
    )
    # neuron_attr: (1, 3, 224, 224)
    # Aggregate across color channels
    nmap = neuron_attr.abs().sum(dim=1).squeeze(0).detach().cpu()
    nmap = (nmap - nmap.min()) / (nmap.max() - nmap.min() + 1e-8)
    neuron_maps.append(nmap.numpy())
    print(f"Channel {c:4d} at ({h},{w}): computed neuron conductance")

In [ ]:
# Visualize: what does each top neuron "see"?
fig, axes = plt.subplots(2, TOP_K + 1, figsize=(24, 8))
fig.suptitle(
    f'Neuron Conductance: What Do Top Neurons See? — {labels[top_class]}',
    fontsize=12
)

# Column 0: input image
for row in range(2):
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title('Input Image', fontsize=9)
    axes[row, 0].axis('off')

# Columns 1–TOP_K: per-neuron maps
for col, ((c, h, w, cond), nmap) in enumerate(
        zip(top_neurons, neuron_maps), start=1):
    # Row 0: neuron attribution heatmap
    im = axes[0, col].imshow(nmap, cmap='hot', vmin=0, vmax=1)
    axes[0, col].set_title(
        f'ch={c}, pos=({h},{w})\ncond={cond:+.4f}',
        fontsize=8
    )
    axes[0, col].axis('off')
    plt.colorbar(im, ax=axes[0, col], fraction=0.046)

    # Row 1: overlay on original image
    axes[1, col].imshow(img_np)
    axes[1, col].imshow(nmap, alpha=0.6, cmap='hot', vmin=0, vmax=1)
    axes[1, col].set_title(f'Overlay ch={c}', fontsize=8)
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

print("""
Each heatmap shows which input pixels caused that specific neuron to activate.
High-conductance neurons in layer4 should respond to semantically meaningful
regions of the image (dog body parts, textures specific to the predicted class).
""")

## 2. Class Selectivity: Dog-Specific vs Shared Neurons

A neuron is **class-selective** if it has high conductance for one class but low conductance for others. A neuron is **shared** if it has high conductance for many classes.

We test this by computing Layer Conductance for two classes (dog and cat) and comparing which neurons are important for each.

In [ ]:
# Compute layer conductance for dog class and cat class (class 281)
dog_cls = top_class
cat_cls = 281  # tabby cat

lc_dog = LayerConductance(model, TARGET_LAYER)
attr_dog = lc_dog.attribute(
    input_tensor, baselines=baseline,
    target=dog_cls, n_steps=40
)

lc_cat = LayerConductance(model, TARGET_LAYER)
attr_cat = lc_cat.attribute(
    input_tensor, baselines=baseline,
    target=cat_cls, n_steps=40
)

# Mean absolute conductance per channel for both classes
dog_channel_cond = attr_dog.abs().mean(dim=(-2, -1)).squeeze().detach().cpu().numpy()
cat_channel_cond = attr_cat.abs().mean(dim=(-2, -1)).squeeze().detach().cpu().numpy()

# Selectivity index: |dog - cat| / (dog + cat + eps)
eps = 1e-8
selectivity = np.abs(dog_channel_cond - cat_channel_cond) / (
    dog_channel_cond + cat_channel_cond + eps
)

# Classify neurons
dog_specific = (selectivity > 0.7) & (dog_channel_cond > cat_channel_cond)
cat_specific = (selectivity > 0.7) & (cat_channel_cond > dog_channel_cond)
shared = selectivity < 0.3

print(f"Channel classification (of 2048 channels):")
print(f"  Dog-selective  (sel>0.7, dog>cat): {dog_specific.sum():4d}")
print(f"  Cat-selective  (sel>0.7, cat>dog): {cat_specific.sum():4d}")
print(f"  Shared         (sel<0.3):          {shared.sum():4d}")
print(f"  Mixed:                             {(2048 - dog_specific.sum() - cat_specific.sum() - shared.sum()):4d}")

In [ ]:
# Scatter plot: dog vs cat conductance per channel
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f'Neuron Selectivity: Dog vs Cat — Layer4 Channel Conductance\n'
    f'Image class: {labels[dog_cls][:30]}',
    fontsize=12
)

# Scatter: dog conductance vs cat conductance per channel
ax = axes[0]
ax.scatter(dog_channel_cond[shared], cat_channel_cond[shared],
            alpha=0.3, c='gray', s=8, label=f'Shared ({shared.sum()})')
ax.scatter(dog_channel_cond[dog_specific], cat_channel_cond[dog_specific],
            alpha=0.7, c='#3498db', s=20, label=f'Dog-selective ({dog_specific.sum()})')
ax.scatter(dog_channel_cond[cat_specific], cat_channel_cond[cat_specific],
            alpha=0.7, c='#e74c3c', s=20, label=f'Cat-selective ({cat_specific.sum()})')
ax.plot([0, max(dog_channel_cond.max(), cat_channel_cond.max())],
         [0, max(dog_channel_cond.max(), cat_channel_cond.max())],
         'k--', alpha=0.3, lw=1, label='Equal importance')
ax.set_xlabel(f'Conductance for "{labels[dog_cls][:20]}"')
ax.set_ylabel(f'Conductance for "{labels[cat_cls][:20]}"')
ax.set_title('Per-Channel Conductance: Dog vs Cat')
ax.legend(fontsize=9)

# Selectivity distribution
axes[1].hist(selectivity, bins=50, color='steelblue', alpha=0.8, edgecolor='none')
axes[1].axvline(0.3, color='gray', linestyle='--', label='Shared threshold')
axes[1].axvline(0.7, color='red',  linestyle='--', label='Selective threshold')
axes[1].set_xlabel('Selectivity Index (|dog-cat|/(dog+cat))')
axes[1].set_ylabel('Number of Channels')
axes[1].set_title('Distribution of Neuron Selectivity')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 3. Visualizing a Dog-Selective vs Shared Neuron

We pick one highly dog-selective neuron and one shared neuron, then use Neuron Conductance to see what each neuron "detects" in the input.

In [ ]:
nc = NeuronConductance(model, TARGET_LAYER)

def get_neuron_map(channel_idx, h=3, w=3):
    """
    Compute Neuron Conductance for a given channel at spatial location (h, w).
    Returns normalized 2D map (224, 224).
    """
    attr = nc.attribute(
        input_tensor,
        neuron_selector=(channel_idx, h, w),
        target=top_class,
        baselines=baseline,
        n_steps=N_STEPS
    )
    nmap = attr.abs().sum(dim=1).squeeze(0).detach().cpu()
    return ((nmap - nmap.min()) / (nmap.max() - nmap.min() + 1e-8)).numpy()


# Find a dog-selective neuron (highest selectivity, dog>cat)
dog_sel_channels = np.where(dog_specific)[0]
if len(dog_sel_channels) > 0:
    # Pick the one with highest dog conductance
    best_dog_ch = dog_sel_channels[
        dog_channel_cond[dog_sel_channels].argmax()
    ]
else:
    best_dog_ch = dog_channel_cond.argmax()

# Find a shared neuron (lowest selectivity, high total conductance)
shared_channels = np.where(shared)[0]
if len(shared_channels) > 0:
    total_cond = dog_channel_cond + cat_channel_cond
    best_shared_ch = shared_channels[
        total_cond[shared_channels].argmax()
    ]
else:
    best_shared_ch = 0

# Find a low-conductance (dormant) neuron
dormant_ch = (dog_channel_cond + cat_channel_cond).argmin()

print(f"Dog-selective channel:  {best_dog_ch} "
      f"(dog cond={dog_channel_cond[best_dog_ch]:.5f}, "
      f"cat cond={cat_channel_cond[best_dog_ch]:.5f})")
print(f"Shared channel:         {best_shared_ch} "
      f"(dog cond={dog_channel_cond[best_shared_ch]:.5f}, "
      f"cat cond={cat_channel_cond[best_shared_ch]:.5f})")
print(f"Dormant channel:        {dormant_ch} "
      f"(dog cond={dog_channel_cond[dormant_ch]:.5f}, "
      f"cat cond={cat_channel_cond[dormant_ch]:.5f})")

# Compute Neuron Conductance maps
# Use center spatial location (3, 3) in the 7×7 feature map
dog_sel_map = get_neuron_map(best_dog_ch)
shared_map  = get_neuron_map(best_shared_ch)
dormant_map = get_neuron_map(dormant_ch)

print("\nNeuron Conductance maps computed.")

In [ ]:
# Visualize: dog-selective vs shared vs dormant neuron
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle(
    'Neuron Conductance: Selective vs Shared vs Dormant Neurons',
    fontsize=12
)

panels = [
    (dog_sel_map, f'Dog-Selective\nch={best_dog_ch}\n'
                  f'sel={selectivity[best_dog_ch]:.2f}',
     '#3498db'),
    (shared_map,  f'Shared\nch={best_shared_ch}\n'
                  f'sel={selectivity[best_shared_ch]:.2f}',
     '#2ecc71'),
    (dormant_map, f'Dormant\nch={dormant_ch}\n'
                  f'sel={selectivity[dormant_ch]:.2f}',
     '#e74c3c'),
]

# Column 0: input image
for row in range(2):
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title('Input Image', fontsize=9)
    axes[row, 0].axis('off')

for col, (nmap, title, color) in enumerate(panels, start=1):
    # Row 0: pure heatmap
    im = axes[0, col].imshow(nmap, cmap='hot', vmin=0, vmax=1)
    axes[0, col].set_title(title, fontsize=9)
    axes[0, col].axis('off')
    plt.colorbar(im, ax=axes[0, col], fraction=0.046)

    # Row 1: overlay
    axes[1, col].imshow(img_np)
    axes[1, col].imshow(nmap, alpha=0.6, cmap='hot', vmin=0, vmax=1)
    axes[1, col].set_title(title.split('\n')[0] + ' Overlay', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

print("""
Expected findings:
- Dog-selective neuron: responds to dog-specific features (fur texture, ear shape, body)
- Shared neuron: responds to features common to many classes (general shapes, backgrounds)
- Dormant neuron: near-uniform map — this neuron is not involved in this prediction
""")

## 4. Exercise: Systematic Neuron Analysis

Now that you know how to identify and visualize important neurons, apply the same analysis systematically.

**Task:**
1. Load the cat image (URL provided below)
2. Find the top-3 neurons by absolute conductance in `layer3` (not layer4)
3. Compute Neuron Conductance maps for those 3 neurons
4. Report how many channels in layer3 are "dormant" (absolute conductance < 0.01% of max)

In [ ]:
# Exercise: Neuron analysis on cat image at layer3

CAT_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg"

# Load cat image
try:
    with urllib.request.urlopen(CAT_URL, timeout=10) as resp:
        cat_raw = Image.open(io.BytesIO(resp.read())).convert('RGB')
except Exception:
    cat_raw = Image.fromarray(np.random.randint(80, 200, (320, 320, 3), dtype=np.uint8))

cat_tensor = preprocess(cat_raw).unsqueeze(0).to(DEVICE)
cat_np_ex = denormalize(cat_tensor.squeeze(0)).permute(1, 2, 0).numpy()
cat_baseline = torch.zeros_like(cat_tensor)

with torch.no_grad():
    cat_class = model(cat_tensor).argmax().item()
print(f"Cat prediction: {labels[cat_class]} (class {cat_class})")

# Step 1: Layer Conductance at layer3 for cat image
LAYER3 = model.layer3[-1]
lc3 = LayerConductance(model, LAYER3)
attr_cat3 = lc3.attribute(
    cat_tensor, baselines=cat_baseline,
    target=cat_class, n_steps=40
)
# attr_cat3: (1, 1024, 14, 14) for ResNet-50 layer3

print(f"Layer3 attribution shape: {attr_cat3.shape}")
C3, H3, W3 = attr_cat3.shape[1], attr_cat3.shape[2], attr_cat3.shape[3]

# Step 2: Find top-3 neurons
attr3_flat = attr_cat3.abs().squeeze(0).view(-1)
top3_flat = attr3_flat.topk(3).indices

top3_neurons = []
for flat_idx in top3_flat:
    flat_idx = flat_idx.item()
    c = flat_idx // (H3 * W3)
    hw = flat_idx % (H3 * W3)
    h = hw // W3
    w = hw % W3
    cond = attr_cat3[0, c, h, w].item()
    top3_neurons.append((c, h, w, cond))

print(f"\nTop-3 neurons in layer3 for cat image:")
for i, (c, h, w, cond) in enumerate(top3_neurons):
    print(f"  #{i+1}: channel={c}, pos=({h},{w}), conductance={cond:+.5f}")

# Step 3: Neuron Conductance for each top-3 neuron
nc3 = NeuronConductance(model, LAYER3)

cat_neuron_maps = []
for c, h, w, cond in top3_neurons:
    attr = nc3.attribute(
        cat_tensor,
        neuron_selector=(c, h, w),
        target=cat_class,
        baselines=cat_baseline,
        n_steps=40
    )
    nmap = attr.abs().sum(dim=1).squeeze(0).detach().cpu()
    nmap = (nmap - nmap.min()) / (nmap.max() - nmap.min() + 1e-8)
    cat_neuron_maps.append(nmap.numpy())

# Step 4: Count dormant neurons
# Channel-level conductance (mean over spatial dimensions)
ch_cond3 = attr_cat3.abs().mean(dim=(-2, -1)).squeeze().detach().cpu().numpy()
threshold_dormant = ch_cond3.max() * 0.0001  # 0.01% of max
n_dormant = (ch_cond3 < threshold_dormant).sum()

print(f"\nLayer3 dormant channel analysis:")
print(f"  Conductance threshold: {threshold_dormant:.7f} (0.01% of max)")
print(f"  Dormant channels: {n_dormant}/{C3} ({n_dormant/C3:.1%})")

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle(
    f'Neuron Conductance at Layer3 — Cat Image ({labels[cat_class][:25]})',
    fontsize=12
)

for row in range(2):
    axes[row, 0].imshow(cat_np_ex)
    axes[row, 0].set_title('Cat Image', fontsize=9)
    axes[row, 0].axis('off')

for col, ((c, h, w, cond), nmap) in enumerate(
        zip(top3_neurons, cat_neuron_maps), start=1):
    im = axes[0, col].imshow(nmap, cmap='hot', vmin=0, vmax=1)
    axes[0, col].set_title(f'ch={c}, pos=({h},{w})\ncond={cond:+.4f}', fontsize=8)
    axes[0, col].axis('off')
    plt.colorbar(im, ax=axes[0, col], fraction=0.046)

    axes[1, col].imshow(cat_np_ex)
    axes[1, col].imshow(nmap, alpha=0.6, cmap='hot', vmin=0, vmax=1)
    axes[1, col].set_title(f'Overlay ch={c}', fontsize=8)
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

# Validation checks
assert len(top3_neurons) == 3, "Should have exactly 3 top neurons"
assert all(0 <= c < C3 for c, h, w, _ in top3_neurons), "Channel indices must be valid"
assert n_dormant >= 0 and n_dormant < C3, "Dormant count must be non-negative and < total"
print("\nAll checks passed.")

## Summary

### What You Learned

1. **Neuron Conductance** returns input attribution for a specific intermediate neuron: "which input pixels caused this neuron to activate?"
2. **Two-step workflow:** Layer Conductance identifies top neurons → Neuron Conductance explains each one
3. **Class selectivity:** neurons can be dog-specific, cat-specific, or shared across classes — selectivity index quantifies this
4. **Sparsity:** most neurons are dormant for any given prediction; typically <20% of channels carry >80% of conductance
5. **Layer depth effect:** layer3 neurons respond to object-level features (parts); layer4 to semantic category features

### Key APIs
```python
from captum.attr import LayerConductance, NeuronConductance

# Find important neurons
lc = LayerConductance(model, model.layer4[-1])
attr = lc.attribute(input_tensor, baselines=baseline, target=cls, n_steps=50)
top_neuron_idx = attr.abs().squeeze(0).view(-1).argmax().item()

# Explain what that neuron sees
nc = NeuronConductance(model, model.layer4[-1])
neuron_attr = nc.attribute(
    input_tensor,
    neuron_selector=(channel, h, w),  # (c, h, w) from above
    target=cls, baselines=baseline, n_steps=50
)
```

### Module 03 Complete

You now have a complete layer attribution toolkit:
- **GradCAM**: fast spatial localization — which regions?
- **Layer Conductance**: which layer decides?
- **Neuron Conductance**: which neurons, and what do they detect?

**Next Module:** Module 04 covers perturbation methods — Occlusion, Feature Ablation, and Shapley Value Sampling. These methods don't use gradients at all, making them applicable to non-differentiable models.